<a href="https://colab.research.google.com/github/MohamedAamer01/otto-recommender-sys/blob/main/notebooks/06_covisitation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, time, pickle
import numpy as np, pandas as pd
from google.colab import drive
drive.mount("/content/drive")

NPY_DIR = "/content/drive/MyDrive/OTTO_Recommender_System/data/processed"
N, N_SESSIONS = 216_716_096, 12_899_779

aid_arr   = np.load(f"{NPY_DIR}/aid.npy")
ts_arr    = np.load(f"{NPY_DIR}/ts.npy")
type_arr  = np.load(f"{NPY_DIR}/type.npy")
sizes_arr = np.load(f"{NPY_DIR}/sizes.npy")
assert len(aid_arr) == N and sizes_arr.sum() == N

with open(f"{NPY_DIR}/dev_validation.pkl", "rb") as f:
    dev = pickle.load(f)
T, GT, prefix_len = dev["T"], dev["GT"], dev["prefix_len"]
cum = np.concatenate([[0], np.cumsum(sizes_arr, dtype=np.int64)])
WEIGHTS = {"clicks": 0.10, "carts": 0.30, "orders": 0.60}

Mounted at /content/drive


In [2]:
from collections import Counter

def covisit_block(aid_b, ts_b, sizes_b, K, window, tmax=None):
    """Paires ordonnées (i, j) des K derniers événements (ts < tmax) de chaque session.
    Retourne (clés triées, compteurs). clé = i * 2**21 + j."""
    S = len(sizes_b)
    sess = np.repeat(np.arange(S), sizes_b)
    if tmax is not None:                              # anti-fuite : on retire les événements >= T
        keep = ts_b < tmax
        sess, aid_b, ts_b = sess[keep], aid_b[keep], ts_b[keep]
    sz = np.bincount(sess, minlength=S)
    starts = np.cumsum(sz) - sz
    pos_end = sz[sess] - 1 - (np.arange(len(sess)) - starts[sess])   # 0 = dernier événement
    sel = pos_end < K                                 # on ne garde que les K derniers
    A  = np.full((S, K), -1, dtype=np.int64)          # matrice sessions x K, -1 = vide
    Tm = np.zeros((S, K), dtype=np.int64)
    A[sess[sel], pos_end[sel]]  = aid_b[sel]
    Tm[sess[sel], pos_end[sel]] = ts_b[sel]
    ai, aj = A[:, :, None], A[:, None, :]             # toutes les paires (i, j) d'une session
    ok = (ai >= 0) & (aj >= 0) & (ai != aj) & (np.abs(Tm[:, :, None] - Tm[:, None, :]) <= window)
    keys = (np.broadcast_to(ai, ok.shape)[ok] << 21) | np.broadcast_to(aj, ok.shape)[ok]
    return np.unique(keys, return_counts=True)

def merge(parts):
    k = np.concatenate([p[0] for p in parts]); c = np.concatenate([p[1] for p in parts])
    u, inv = np.unique(k, return_inverse=True)
    return u, np.bincount(inv, weights=c).astype(np.int64)

# --- test contre un calcul brut sur petites données ---
rng = np.random.default_rng(1)
sz = rng.integers(2, 30, 400).astype(np.int32)
a_ = rng.integers(0, 40, sz.sum()).astype(np.int32)
t_ = np.concatenate([np.sort(rng.integers(0, 200, n)) for n in sz]).astype(np.int32)

def brute(aid, ts, sizes, K, window, tmax):
    C, p = Counter(), 0
    for n in sizes:
        a, t = aid[p:p+n], ts[p:p+n]; p += n
        m = t < tmax; a, t = a[m][-K:], t[m][-K:]
        for x in range(len(a)):
            for y in range(len(a)):
                if x != y and a[x] != a[y] and abs(int(t[x]) - int(t[y])) <= window:
                    C[(int(a[x]), int(a[y]))] += 1
    return C

for K, W, tm in [(5, 50, 150), (10, 1000, 10**9)]:
    k, c = covisit_block(a_, t_, sz, K, W, tm)
    got = {(int(x >> 21), int(x & ((1 << 21) - 1))): int(v) for x, v in zip(k, c)}
    assert got == dict(brute(a_, t_, sz, K, W, tm)), (K, W, tm)
print("covisit_block conforme au calcul brut")

covisit_block conforme au calcul brut


In [3]:
BLOCK = 10_000

def sample_covisit(n_blocks, K=10, window=86400, tmax=T):
    starts = np.linspace(0, N_SESSIONS - BLOCK, n_blocks).astype(np.int64)
    parts = []
    for s in starts:
        a, z = cum[s], cum[s + BLOCK]
        parts.append(covisit_block(aid_arr[a:z], ts_arr[a:z], sizes_arr[s:s + BLOCK], K, window, tmax))
    return merge(parts)

res = {}
for n_blocks in (10, 20):
    t0 = time.time()
    keys, counts = sample_covisit(n_blocks)
    res[n_blocks] = len(keys)
    print(f"{n_blocks * BLOCK:>7,} sessions | paires brutes {counts.sum():>12,} | paires distinctes {len(keys):>11,} "
          f"| {(keys.nbytes + counts.nbytes)/1e6:6.0f} MB | {time.time()-t0:.0f}s")
print(f"Croissance des paires distinctes quand on double les sessions : x{res[20]/res[10]:.2f}")

100,000 sessions | paires brutes    2,088,546 | paires distinctes   1,220,760 |     20 MB | 1s
200,000 sessions | paires brutes    4,334,040 | paires distinctes   2,441,810 |     39 MB | 1s
Croissance des paires distinctes quand on double les sessions : x2.00


constructeur partitionné, avec test

In [4]:
def build_topn(aid_arr, ts_arr, sizes_arr, cum, n_items, K=10, window=86400, tmax=None,
               N=50, P=4, BLOCK=50_000):
    """Table des N meilleurs voisins par article. nbr_j[i] = voisins, nbr_c[i] = compteurs."""
    nbr_j = np.full((n_items, N), -1, dtype=np.int32)
    nbr_c = np.zeros((n_items, N), dtype=np.float32)
    n_sess, MASK = len(sizes_arr), (1 << 21) - 1
    for p in range(P):
        t0, parts = time.time(), []
        for s in range(0, n_sess, BLOCK):
            e = min(s + BLOCK, n_sess); a, z = cum[s], cum[e]
            k, c = covisit_block(aid_arr[a:z], ts_arr[a:z], sizes_arr[s:e], K, window, tmax)
            m = ((k >> 21) % P) == p                  # on ne garde que la partition p
            parts.append((k[m], c[m]))
            if len(parts) >= 20:
                parts = [merge(parts)]                # fusion progressive : limite la mémoire
        keys, counts = merge(parts)
        i = (keys >> 21).astype(np.int64); j = (keys & MASK).astype(np.int32)
        order = np.lexsort((-counts, i))              # par article, puis compteur décroissant
        i, j, counts = i[order], j[order], counts[order]
        rank = np.arange(len(i)) - np.searchsorted(i, i, side="left")
        keep = rank < N
        nbr_j[i[keep], rank[keep]] = j[keep]
        nbr_c[i[keep], rank[keep]] = counts[keep]
        print(f"partition {p+1}/{P} : {len(keys):,} paires distinctes | {time.time()-t0:.0f}s")
        del keys, counts, i, j, order, parts
    return nbr_j, nbr_c

# --- test : partitionné == non partitionné, et conforme à un calcul direct ---
n_it = int(a_.max()) + 1
cs_ = np.concatenate([[0], np.cumsum(sz, dtype=np.int64)])
nj_a, nc_a = build_topn(a_, t_, sz, cs_, n_it, K=10, window=1000, tmax=10**9, N=5, P=4, BLOCK=100)
nj_b, nc_b = build_topn(a_, t_, sz, cs_, n_it, K=10, window=1000, tmax=10**9, N=5, P=1, BLOCK=100)
assert np.array_equal(nj_a, nj_b) and np.array_equal(nc_a, nc_b)
print("Partitions : table identique à la version non partitionnée")

partition 1/4 : 390 paires distinctes | 0s
partition 2/4 : 390 paires distinctes | 0s
partition 3/4 : 390 paires distinctes | 0s
partition 4/4 : 390 paires distinctes | 0s
partition 1/1 : 1,560 paires distinctes | 0s
Partitions : table identique à la version non partitionnée


construction sur tes données (avant T) et sauvegarde

In [5]:
n_items = int(aid_arr.max()) + 1
t0 = time.time()
nbr_j, nbr_c = build_topn(aid_arr, ts_arr, sizes_arr, cum, n_items,
                          K=10, window=86400, tmax=T, N=50, P=4)
print(f"Construction : {(time.time()-t0)/60:.1f} min | table : {(nbr_j.nbytes + nbr_c.nbytes)/1e9:.2f} GB")
print(f"Articles avec au moins 1 voisin : {(nbr_j[:, 0] >= 0).mean()*100:.1f} %")

np.save(f"{NPY_DIR}/covisit_K10_T_nbr_j.npy", nbr_j)
np.save(f"{NPY_DIR}/covisit_K10_T_nbr_c.npy", nbr_c)

partition 1/4 : 26,992,123 paires distinctes | 90s
partition 2/4 : 26,904,327 paires distinctes | 85s
partition 3/4 : 26,888,333 paires distinctes | 87s
partition 4/4 : 26,777,917 paires distinctes | 85s
Construction : 5.8 min | table : 0.74 GB
Articles avec au moins 1 voisin : 91.5 %


évaluation de B4 contre B3, avec bootstrap apparié

In [6]:
def top_lists(mask, k=40):
    return {t: np.argsort(-np.bincount(aid_arr[mask & (type_arr == i)], minlength=n_items),
                          kind="stable")[:k].tolist()
            for t, i in {"clicks": 0, "carts": 1, "orders": 2}.items()}
tops_T = top_lists(ts_arr < T)

def seen_items(sid):                                  # articles du préfixe, récents d'abord
    a, k = cum[sid], prefix_len[sid]
    return list(dict.fromkeys(map(int, aid_arr[a:a + k][::-1])))

def topup(base, top, n=20):
    lst = list(base); s = set(lst)
    for x in top:
        if len(lst) == n: break
        if x not in s: lst.append(x)
    return lst

def lists_b3(seen, tops, n=20):
    return {t: topup(seen[:n], tops[t], n) for t in WEIGHTS}

def lists_b4(seen, tops, nj, nc, n_seed=10, n=20):
    base = seen[:n]; sset = set(base)
    if len(base) < n:
        sd = np.asarray(seen[:n_seed])
        js, cs = nj[sd].ravel(), nc[sd].ravel(); m = js >= 0
        if m.any():
            u, inv = np.unique(js[m], return_inverse=True)
            sc = np.bincount(inv, weights=cs[m])      # somme des compteurs sur les graines
            order = np.argsort(-sc, kind="stable")
            base = base + [int(x) for x in u[order] if int(x) not in sset][:n - len(base)]
    return {t: topup(base, tops[t], n) for t in WEIGHTS}

def per_session(make, GT, k=20):
    sids = list(GT)
    H = {t: np.zeros(len(sids)) for t in WEIGHTS}; D = {t: np.zeros(len(sids)) for t in WEIGHTS}
    for i, sid in enumerate(sids):
        lists = make(sid)
        for t in WEIGHTS:
            gt = GT[sid][t]
            if gt:
                H[t][i] = len(set(lists[t][:k]) & gt); D[t][i] = min(k, len(gt))
    return H, D

def score_from(H, D, idx=None):
    out = {}
    for t in WEIGHTS:
        h = H[t] if idx is None else H[t][idx]; d = D[t] if idx is None else D[t][idx]
        out[t] = h.sum() / d.sum()
    out["score"] = sum(WEIGHTS[t] * out[t] for t in WEIGHTS)
    return out

def boot_diff_all(HA, HB, D, B=200, seed=0):
    r = np.random.default_rng(seed); n = len(next(iter(D.values())))
    diffs = {k: [] for k in list(WEIGHTS) + ["score"]}
    for _ in range(B):
        idx = r.integers(0, n, n); a = score_from(HA, D, idx); b = score_from(HB, D, idx)
        for k in diffs: diffs[k].append(b[k] - a[k])
    return {k: np.percentile(v, [2.5, 97.5]) for k, v in diffs.items()}

t0 = time.time()
H3, D = per_session(lambda s: lists_b3(seen_items(s), tops_T), GT)
r3 = score_from(H3, D)
assert abs(r3["score"] - 0.4706) < 5e-4, "B3 ne redonne pas 0.4706 : ne pas continuer"
H4, _ = per_session(lambda s: lists_b4(seen_items(s), tops_T, nbr_j, nbr_c), GT)
r4 = score_from(H4, D)
print(f"Évaluation : {(time.time()-t0)/60:.1f} min")

display(pd.DataFrame({"B3": r3, "B4": r4}).T.round(4))
for k, (lo, hi) in boot_diff_all(H3, H4, D).items():
    print(f"B4 - B3 | {k:<7}: IC95 [{lo:+.4f}, {hi:+.4f}]")

Évaluation : 0.7 min


,clicks,carts,orders,score
B3,0.3275,0.3121,0.5738,0.4706
B4,0.5028,0.3947,0.6235,0.5428


B4 - B3 | clicks : IC95 [+0.1740, +0.1769]
B4 - B3 | carts  : IC95 [+0.0806, +0.0845]
B4 - B3 | orders : IC95 [+0.0478, +0.0515]
B4 - B3 | score  : IC95 [+0.0706, +0.0736]


In [7]:
def lists_b4x(seen, tops, nj, nc, n_seed=10, gamma=1.0, row_norm=False, n=20):
    base = seen[:n]; sset = set(base)
    if len(base) < n:
        sd = np.asarray(seen[:n_seed])
        cs = nc[sd].astype(np.float64)                          # (graines, 50)
        if row_norm:                                            # chaque graine vote pour une distribution
            cs = cs / np.maximum(cs.sum(axis=1, keepdims=True), 1)
        cs = cs * (gamma ** np.arange(len(sd)))[:, None]        # décroissance selon la récence
        js = nj[sd].ravel(); cs = cs.ravel(); m = js >= 0
        if m.any():
            u, inv = np.unique(js[m], return_inverse=True)
            sc = np.bincount(inv, weights=cs[m])
            order = np.argsort(-sc, kind="stable")
            base = base + [int(x) for x in u[order] if int(x) not in sset][:n - len(base)]
    return {t: topup(base, tops[t], n) for t in WEIGHTS}

# contrôle : avec les paramètres par défaut, on retrouve B4
assert abs(score_from(per_session(lambda s: lists_b4x(seen_items(s), tops_T, nbr_j, nbr_c), GT)[0], D)["score"]
           - r4["score"]) < 1e-9
print("lists_b4x (défauts) == B4")

cfgs = {"B4 (réf.)": {},
        "n_seed=1": dict(n_seed=1), "n_seed=3": dict(n_seed=3), "n_seed=20": dict(n_seed=20),
        "gamma=0.7": dict(gamma=0.7), "gamma=0.5": dict(gamma=0.5),
        "row_norm": dict(row_norm=True)}
rows, HH = {}, {}
for name, kw in cfgs.items():
    t0 = time.time()
    H, _ = per_session(lambda s, kw=kw: lists_b4x(seen_items(s), tops_T, nbr_j, nbr_c, **kw), GT)
    HH[name] = H; rows[name] = score_from(H, D)
    print(f"{name:<12} {time.time()-t0:.0f}s")
res = pd.DataFrame(rows).T
res["Δ score vs B4"] = res["score"] - res.loc["B4 (réf.)", "score"]
display(res.round(4))

for name in cfgs:
    if name == "B4 (réf.)": continue
    lo, hi = boot_diff_all(HH["B4 (réf.)"], HH[name], D, B=100)["score"]
    print(f"{name:<12} - B4 : IC95 [{lo:+.4f}, {hi:+.4f}]")

lists_b4x (défauts) == B4
B4 (réf.)    40s
n_seed=1     30s
n_seed=3     34s
n_seed=20    39s
gamma=0.7    36s
gamma=0.5    38s
row_norm     41s


,clicks,carts,orders,score,Δ score vs B4
B4 (réf.),0.5028,0.3947,0.6235,0.5428,0.0000
n_seed=1,0.5071,0.3984,0.6237,0.5445,0.0017
n_seed=3,0.5065,0.3982,0.6253,0.5453,0.0024
n_seed=20,0.5026,0.3943,0.6234,0.5426,-0.0002
gamma=0.7,0.5068,0.3989,0.6260,0.5459,0.0031
gamma=0.5,0.5091,0.4010,0.6270,0.5474,0.0046
row_norm,0.5074,0.3995,0.6269,0.5467,0.0039


n_seed=1     - B4 : IC95 [+0.0005, +0.0025]
n_seed=3     - B4 : IC95 [+0.0018, +0.0030]
n_seed=20    - B4 : IC95 [-0.0004, -0.0001]
gamma=0.7    - B4 : IC95 [+0.0027, +0.0036]
gamma=0.5    - B4 : IC95 [+0.0040, +0.0052]
row_norm     - B4 : IC95 [+0.0034, +0.0046]


Combiner et passer a la table

In [8]:
cfgs2 = {"γ=0.5 (réf.)":       dict(gamma=0.5),
         "γ=0.3":              dict(gamma=0.3),
         "γ=0.5 + row_norm":   dict(gamma=0.5, row_norm=True),
         "γ=0.3 + row_norm":   dict(gamma=0.3, row_norm=True),
         "γ=0.7 + row_norm":   dict(gamma=0.7, row_norm=True)}
rows2, HH2 = {}, {}
for name, kw in cfgs2.items():
    t0 = time.time()
    H, _ = per_session(lambda s, kw=kw: lists_b4x(seen_items(s), tops_T, nbr_j, nbr_c, **kw), GT)
    HH2[name] = H; rows2[name] = score_from(H, D)
    print(f"{name:<18} {time.time()-t0:.0f}s")
res2 = pd.DataFrame(rows2).T
res2["Δ vs B4"] = res2["score"] - r4["score"]
res2["Δ vs γ=0.5"] = res2["score"] - res2.loc["γ=0.5 (réf.)", "score"]
display(res2.round(4))

for name in cfgs2:
    if name == "γ=0.5 (réf.)": continue
    lo, hi = boot_diff_all(HH2["γ=0.5 (réf.)"], HH2[name], D, B=100)["score"]
    print(f"{name:<18} - (γ=0.5) : IC95 [{lo:+.4f}, {hi:+.4f}]")

γ=0.5 (réf.)       38s
γ=0.3              38s
γ=0.5 + row_norm   42s
γ=0.3 + row_norm   45s
γ=0.7 + row_norm   42s


,clicks,carts,orders,score,Δ vs B4,Δ vs γ=0.5
γ=0.5 (réf.),0.5091,0.4010,0.6270,0.5474,0.0046,0.0000
γ=0.3,0.5106,0.4017,0.6274,0.5480,0.0052,0.0006
γ=0.5 + row_norm,0.5134,0.4042,0.6285,0.5497,0.0069,0.0023
γ=0.3 + row_norm,0.5126,0.4033,0.6272,0.5486,0.0058,0.0012
γ=0.7 + row_norm,0.5122,0.4041,0.6288,0.5498,0.0070,0.0023


γ=0.3              - (γ=0.5) : IC95 [+0.0002, +0.0011]
γ=0.5 + row_norm   - (γ=0.5) : IC95 [+0.0017, +0.0028]
γ=0.3 + row_norm   - (γ=0.5) : IC95 [+0.0004, +0.0018]
γ=0.7 + row_norm   - (γ=0.5) : IC95 [+0.0017, +0.0030]


**Journal des resulatats**

In [9]:
import os, pandas as pd
BASE = "/content/drive/MyDrive/OTTO_Recommender_System"
RESULTS_PATH = f"{BASE}/reports/results.csv"

def log_result(name, r, notes="", split="dev300k", path=RESULTS_PATH):
    row = {"model": name, "split": split, "clicks": round(float(r["clicks"]), 4),
           "carts": round(float(r["carts"]), 4), "orders": round(float(r["orders"]), 4),
           "score": round(float(r["score"]), 4), "notes": notes}
    new = pd.DataFrame([row])
    if os.path.exists(path):
        old = pd.read_csv(path)
        old = old[~((old["model"] == name) & (old["split"] == split))]   # remplace la ligne du même modèle
        new = pd.concat([old, new], ignore_index=True)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    new.to_csv(path, index=False)
    return new

# Historique, avec les valeurs déjà mesurées (dev de 300 000 sessions)
history = [
    ("B0 top-20 global",              (0.0082, 0.0062, 0.0047, 0.0055), "popularité avant T"),
    ("B1 top-20 par type",            (0.0082, 0.0078, 0.0060, 0.0067), "popularité par type"),
    ("B2 dernière semaine",           (0.0075, 0.0068, 0.0069, 0.0070), "pas significatif vs B1"),
    ("Plafond session seulement",     (0.3239, 0.3139, 0.5959, 0.4841), "classement parfait, sans complément"),
    ("B3 session + popularité",       (0.3275, 0.3121, 0.5738, 0.4706), "articles du préfixe, récents d'abord"),
    ("B4 co-visitation K=10 top-50",  (0.5028, 0.3947, 0.6235, 0.5428), "fenêtre 24 h, 10 graines"),
    ("B4 γ=0.5",                      (0.5091, 0.4010, 0.6270, 0.5474), "décroissance de récence"),
    ("B4* γ=0.5 + normalisation",     (0.5134, 0.4042, 0.6285, 0.5497), "modèle figé après 6.2a"),
]
for name, (c, k, o, s), note in history:
    log_result(name, dict(clicks=c, carts=k, orders=o, score=s), note)

nan = float("nan")
for split, s in [("kaggle_public", 0.47911), ("kaggle_private", 0.48035)]:
    log_result("B3 session + popularité", dict(clicks=nan, carts=nan, orders=nan, score=s),
               "soumission Kaggle (tardive)", split=split)
display(pd.read_csv(RESULTS_PATH))

,model,split,clicks,carts,orders,score,notes
0,B0 top-20 global,dev300k,0.0082,0.0062,0.0047,0.0055,popularité avant T
1,B1 top-20 par type,dev300k,0.0082,0.0078,0.0060,0.0067,popularité par type
2,B2 dernière semaine,dev300k,0.0075,0.0068,0.0069,0.0070,pas significatif vs B1
3,Plafond session seulement,dev300k,0.3239,0.3139,0.5959,0.4841,"classement parfait, sans complément"
4,B3 session + popularité,dev300k,0.3275,0.3121,0.5738,0.4706,"articles du préfixe, récents d'abord"
5,B4 co-visitation K=10 top-50,dev300k,0.5028,0.3947,0.6235,0.5428,"fenêtre 24 h, 10 graines"
6,B4 γ=0.5,dev300k,0.5091,0.4010,0.6270,0.5474,décroissance de récence
7,B4* γ=0.5 + normalisation,dev300k,0.5134,0.4042,0.6285,0.5497,modèle figé après 6.2a
8,B3 session + popularité,kaggle_public,NaN,NaN,NaN,0.4791,soumission Kaggle (tardive)
9,B3 session + popularité,kaggle_private,NaN,NaN,NaN,0.4803,soumission Kaggle (tardive)


**6.b : normaliser le candidat par sa popularité, avant l'élagage**

Constructeur multi-α, avec test

In [10]:
def build_topn_multi(aid_arr, ts_arr, sizes_arr, cum, n_items, pop, alphas=(0.0,), K=10,
                     window=86400, tmax=None, N=50, P=4, BLOCK=50_000):
    """Une table top-N par valeur de alpha, en un seul parcours des données."""
    tables = {a: (np.full((n_items, N), -1, dtype=np.int32),
                  np.zeros((n_items, N), dtype=np.float32)) for a in alphas}
    n_sess, MASK = len(sizes_arr), (1 << 21) - 1
    for p in range(P):
        t0, parts = time.time(), []
        for s in range(0, n_sess, BLOCK):
            e = min(s + BLOCK, n_sess); a, z = cum[s], cum[e]
            k, c = covisit_block(aid_arr[a:z], ts_arr[a:z], sizes_arr[s:e], K, window, tmax)
            m = ((k >> 21) % P) == p
            parts.append((k[m], c[m]))
            if len(parts) >= 20:
                parts = [merge(parts)]
        keys, counts = merge(parts)
        i = (keys >> 21).astype(np.int64); j = (keys & MASK).astype(np.int64)
        start = np.searchsorted(i, i, side="left")      # i est trié : début du groupe de chaque ligne
        rank = np.arange(len(i)) - start                # rang de chaque ligne dans son groupe (inchangé par le tri)
        for al in alphas:
            score = counts / pop[j] ** al               # normalisation par la popularité du candidat
            order = np.lexsort((-score, i))             # par article, puis score décroissant
            keep = rank < N
            nj, nc = tables[al]
            nj[i[keep], rank[keep]] = j[order][keep]
            nc[i[keep], rank[keep]] = score[order][keep]
        print(f"partition {p+1}/{P} : {len(keys):,} paires distinctes | {time.time()-t0:.0f}s")
        del keys, counts, i, j, parts
    return tables

# --- test : partitionné == non partitionné ; alpha=0 == ancienne table ---
n_it = int(a_.max()) + 1
pop_t = np.maximum(np.bincount(a_, minlength=n_it), 1).astype(np.float64)
tA = build_topn_multi(a_, t_, sz, cs_, n_it, pop_t, alphas=(0.0, 0.5), K=10, window=1000, tmax=10**9, N=5, P=4, BLOCK=100)
tB = build_topn_multi(a_, t_, sz, cs_, n_it, pop_t, alphas=(0.0, 0.5), K=10, window=1000, tmax=10**9, N=5, P=1, BLOCK=100)
ref_j, ref_c = build_topn(a_, t_, sz, cs_, n_it, K=10, window=1000, tmax=10**9, N=5, P=1, BLOCK=100)
for al in (0.0, 0.5):
    assert np.array_equal(tA[al][0], tB[al][0]) and np.allclose(tA[al][1], tB[al][1])
assert np.array_equal(tA[0.0][0], ref_j) and np.allclose(tA[0.0][1], ref_c)
print("Multi-alpha : conforme (partitions, alpha=0 == ancienne table)")

partition 1/4 : 390 paires distinctes | 0s
partition 2/4 : 390 paires distinctes | 0s
partition 3/4 : 390 paires distinctes | 0s
partition 4/4 : 390 paires distinctes | 0s
partition 1/1 : 1,560 paires distinctes | 0s
partition 1/1 : 1,560 paires distinctes | 0s
Multi-alpha : conforme (partitions, alpha=0 == ancienne table)


construction sur tes données (≈6 à 7 min)

In [11]:
pop = np.maximum(np.bincount(aid_arr[ts_arr < T], minlength=n_items), 1).astype(np.float64)

t0 = time.time()
tables = build_topn_multi(aid_arr, ts_arr, sizes_arr, cum, n_items, pop,
                          alphas=(0.25, 0.5, 1.0), K=10, window=86400, tmax=T, N=50, P=4)
tables[0.0] = (nbr_j, nbr_c)                  # table de référence, déjà en mémoire
print(f"Construction : {(time.time()-t0)/60:.1f} min")

partition 1/4 : 26,992,123 paires distinctes | 139s
partition 2/4 : 26,904,327 paires distinctes | 126s
partition 3/4 : 26,888,333 paires distinctes | 126s
partition 4/4 : 26,777,917 paires distinctes | 124s
Construction : 8.6 min


évaluation (≈45 s par α)

In [12]:
BEST = dict(gamma=0.5, row_norm=True)            # B4*, figé après 6.2a
rows, HH = {}, {}
for al in sorted(tables):
    nj_a, nc_a = tables[al]
    t0 = time.time()
    H, _ = per_session(lambda s: lists_b4x(seen_items(s), tops_T, nj_a, nc_a, **BEST), GT)
    HH[al] = H; rows[f"alpha={al}"] = score_from(H, D)
    print(f"alpha={al}: {time.time()-t0:.0f}s")

assert abs(rows["alpha=0.0"]["score"] - 0.5497) < 5e-4, "alpha=0 doit redonner B4* (0.5497)"
res = pd.DataFrame(rows).T
res["Δ vs alpha=0"] = res["score"] - res.loc["alpha=0.0", "score"]
display(res.round(4))

for al in sorted(tables):
    if al == 0.0: continue
    d = boot_diff_all(HH[0.0], HH[al], D, B=100)
    print(f"alpha={al}: Δ score IC95 [{d['score'][0]:+.4f}, {d['score'][1]:+.4f}] | "
          f"clicks {d['clicks'].mean():+.4f} | carts {d['carts'].mean():+.4f} | orders {d['orders'].mean():+.4f}")

# Journal de résultats (la plus performante seulement si elle bat B4*)
best_al = max(rows, key=lambda k: rows[k]["score"])
log_result(f"6.2b {best_al}", rows[best_al], "normalisation par la popularité du candidat avant top-N")

alpha=0.0: 43s
alpha=0.25: 42s
alpha=0.5: 43s
alpha=1.0: 43s


,clicks,carts,orders,score,Δ vs alpha=0
alpha=0.0,0.5134,0.4042,0.6285,0.5497,0.0000
alpha=0.25,0.5086,0.4042,0.6284,0.5491,-0.0006
alpha=0.5,0.4816,0.3914,0.6204,0.5378,-0.0119
alpha=1.0,0.3694,0.3327,0.5858,0.4882,-0.0615


alpha=0.25: Δ score IC95 [-0.0011, +0.0001] | clicks -0.0048 | carts -0.0001 | orders -0.0001
alpha=0.5: Δ score IC95 [-0.0127, -0.0112] | clicks -0.0319 | carts -0.0128 | orders -0.0082
alpha=1.0: Δ score IC95 [-0.0629, -0.0600] | clicks -0.1441 | carts -0.0715 | orders -0.0428


,model,split,clicks,carts,orders,score,notes
0,B0 top-20 global,dev300k,0.0082,0.0062,0.0047,0.0055,popularité avant T
1,B1 top-20 par type,dev300k,0.0082,0.0078,0.0060,0.0067,popularité par type
2,B2 dernière semaine,dev300k,0.0075,0.0068,0.0069,0.0070,pas significatif vs B1
3,Plafond session seulement,dev300k,0.3239,0.3139,0.5959,0.4841,"classement parfait, sans complément"
4,B3 session + popularité,dev300k,0.3275,0.3121,0.5738,0.4706,"articles du préfixe, récents d'abord"
5,B4 co-visitation K=10 top-50,dev300k,0.5028,0.3947,0.6235,0.5428,"fenêtre 24 h, 10 graines"
6,B4 γ=0.5,dev300k,0.5091,0.4010,0.6270,0.5474,décroissance de récence
7,B4* γ=0.5 + normalisation,dev300k,0.5134,0.4042,0.6285,0.5497,modèle figé après 6.2a
8,B3 session + popularité,kaggle_public,NaN,NaN,NaN,0.4791,soumission Kaggle (tardive)
9,B3 session + popularité,kaggle_private,NaN,NaN,NaN,0.4803,soumission Kaggle (tardive)


**6.c :**